# Urban Pulse — Modeling: Random Forest

> **File**: `randomforest.ipynb`  
> **Tujuan**: Melatih model klasifikasi slum/non-slum menggunakan **Random Forest**, melakukan **threshold tuning** untuk prioritas recall, menghasilkan **local SHAP** untuk menjelaskan kontribusi fitur pada prediksi per area, dan **mengekspor artefak** (`.joblib` + `model_rf_metadata.json`) yang siap dipakai.

---

## 🔬 Penjelasan Setiap Cell

Notebook ini didesain selaras dengan pipeline evaluasi pada model XGBoost proyek ini. Evaluasi menggunakan Leave-One-City-Out (LOCO) sebagai metrik utama, dan random stratified split (80/20) untuk hyperparameter tuning dan thresholding.

### Cell 1 — Import Library

**Tujuan**: Mengimpor semua library yang dibutuhkan untuk pemodelan Random Forest, evaluasi metrik, dan analisis interpretabilitas (SHAP).

| Library | Kegunaan |
|---|---|
| `joblib` | Serialisasi model Random Forest ke format `.joblib` |
| `sklearn.ensemble.RandomForestClassifier` | Algoritma ensemble bagging utama untuk klasifikasi biner |
| `sklearn.metrics` | Penghitungan metrik klasifikasi (Recall, Precision, F1, ROC-AUC) |
| `shap` | Penghitungan SHAP values untuk interpretasi global dan lokal |

In [1]:
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    recall_score, precision_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
)
from sklearn.ensemble import RandomForestClassifier
import shap

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
RANDOM_STATE = 42

C:\Users\Fauzi\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Cell 2 — Konfigurasi

**Tujuan**: Mendefinisikan konstanta konfigurasi path file data, path penyimpanan model, target kolom label, serta target batas recall untuk threshold tuning.

**Poin Penting**:
- Target recall minimal ditentukan sebesar `0.80` untuk memprioritaskan penyaringan area kumuh (meminimalkan false negative).

In [2]:
PROCESSED_DIR = Path("data/processed")
DATA_PATH = PROCESSED_DIR / "features_buildings_clean.csv"
FEATURE_COLS_PATH = PROCESSED_DIR / "feature_columns.csv"

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / "slum_rf_pipeline.joblib"
METADATA_PATH = MODELS_DIR / "model_rf_metadata.json"

TARGET_COL = "slum"
CITY_COL = "city"
TEST_SIZE = 0.2
TARGET_RECALL = 0.80

### Cell 3 — Load Data

**Tujuan**: Membaca dataset bersih hasil preprocessing (`features_buildings_clean.csv`) beserta daftar 41 kolom fitur pilihan (`feature_columns.csv`).

**Output yang diharapkan**:
- Menampilkan dimensi dataset (830 baris, 49 kolom).
- Menampilkan distribusi jumlah kelas slum (118) dan non-slum (712) beserta persentase slum per kota.

In [3]:
df = pd.read_csv(DATA_PATH)
feature_cols = pd.read_csv(FEATURE_COLS_PATH)["feature_name"].tolist()

print(f"Dataset: {df.shape}")
print(f"Jumlah fitur model: {len(feature_cols)}")
print(f"Distribusi kelas: slum={int(df[TARGET_COL].sum())} "
      f"({df[TARGET_COL].mean()*100:.1f}%) / "
      f"non-slum={int((df[TARGET_COL]==0).sum())}")

df[[CITY_COL, TARGET_COL]].groupby(CITY_COL)[TARGET_COL].agg(["count", "sum", "mean"]) \
    .rename(columns={"count": "total", "sum": "slum", "mean": "pct_slum"})

Dataset: (830, 49)
Jumlah fitur model: 41
Distribusi kelas: slum=118 (14.2%) / non-slum=712


,total,slum,pct_slum
city,,,
ambon,50,15,0.300000
dki,261,62,0.237548
kebumen,460,27,0.058696
samarinda,59,14,0.237288


### Cell 4 — Definisi Model Random Forest

**Tujuan**: Mendefinisikan fungsi pembentuk model `RandomForestClassifier` dengan konfigurasi hyperparameter *best practice* untuk data kecil dan imbalanced.

**Alasan Pemilihan Parameter**:
- `class_weight='balanced_subsample'`: Menghitung ulang bobot kelas secara independen di setiap sampel bootstrap pohon (tree bagging), memberikan penanganan imbalance yang sangat optimal.
- `max_depth=6` dan `min_samples_leaf=3`: Membatasi kedalaman pohon dan jumlah minimal sampel pada daun untuk meredam overfitting spuria pada dataset berukuran kecil.
- `max_features='sqrt'`: Mengurangi korelasi antar pohon dengan hanya mencari split pada $\sqrt{D}$ fitur acak.

In [4]:
def make_rf(random_state=RANDOM_STATE):
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=random_state,
        n_jobs=-1,
    )

### Cell 5 — Evaluasi LOCO (Leave-One-City-Out) — Metrik Utama

**Tujuan**: Melakukan cross-validation berbasis kota (LOCO) dengan threshold default 0.5 untuk mengetahui baseline performa generalisasi model pada kota baru.

**Output yang diharapkan**:
- Tabel hasil evaluasi 4 kota folds berisi metrik Recall, Precision, F1, dan ROC-AUC beserta nilai rata-rata (*mean*) dan standar deviasinya (*std*).

In [5]:
def run_loco_evaluation(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    cities = sorted(df[CITY_COL].unique())
    rows = []

    for city in cities:
        # Baris kota ini jadi TEST, sisanya (3 kota lain) jadi TRAIN
        train_mask = df[CITY_COL] != city
        test_mask = df[CITY_COL] == city

        X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET_COL]
        X_test, y_test = df.loc[test_mask, feature_cols], df.loc[test_mask, TARGET_COL]

        model = make_rf()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        try:
            auc = roc_auc_score(y_test, y_proba)
        except ValueError:
            auc = np.nan

        rows.append({
            "held_out_city": city,
            "n_test": len(y_test),
            "n_slum_test": int(y_test.sum()),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": auc,
        })

    return pd.DataFrame(rows)

loco_results = run_loco_evaluation(df, feature_cols)
print(loco_results)
print("\nRingkasan rata-rata LOCO (threshold default 0.5):")
loco_summary = loco_results[["recall", "precision", "f1", "roc_auc"]].agg(["mean", "std"]).round(3)
loco_summary

  held_out_city  n_test  n_slum_test    recall  precision        f1   roc_auc
0         ambon      50           15  0.533333   0.571429  0.551724  0.847619
1           dki     261           62  0.935484   0.233871  0.374194  0.514184
2       kebumen     460           27  0.000000   0.000000  0.000000  0.775939
3     samarinda      59           14  0.785714   0.323529  0.458333  0.712698

Ringkasan rata-rata LOCO (threshold default 0.5):


,recall,precision,f1,roc_auc
mean,0.564,0.282,0.346,0.713
std,0.411,0.236,0.242,0.143


### Cell 6 — Random Stratified Split & Threshold Tuning

**Tujuan**: Membagi data secara stratified (80/20) berdasarkan kota dan kelas, lalu melakukan pencarian (*scan*) nilai threshold probabilitas terbesar yang dapat menghasilkan nilai Recall kelas slum $\ge 0.80$ pada set testing.

**Output yang diharapkan**:
- Nilai threshold terpilih (misal: 0.28) beserta preview performa model di sekitar threshold tersebut.

In [6]:
strat_key = df[CITY_COL].astype(str) + "_" + df[TARGET_COL].astype(str)
X = df[feature_cols]
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=strat_key
)

rf_split = make_rf().fit(X_train, y_train)
y_pred_split = rf_split.predict(X_test)
y_proba_split = rf_split.predict_proba(X_test)[:, 1]

def find_threshold_for_recall(y_true, y_proba, target_recall=TARGET_RECALL):
    """Cari threshold terbesar yang masih mencapai recall >= target_recall."""
    thresholds = np.linspace(0.01, 0.99, 99)
    best_threshold, best_precision = 0.0, -1.0
    rows = []
    for t in thresholds:
        pred = (y_proba >= t).astype(int)
        r = recall_score(y_true, pred, zero_division=0)
        p = precision_score(y_true, pred, zero_division=0)
        rows.append({"threshold": round(t, 3), "recall": round(r, 3), "precision": round(p, 3)})
        if r >= target_recall and p > best_precision:
            best_threshold, best_precision = t, p
    return best_threshold, pd.DataFrame(rows)

best_threshold_split, threshold_scan_split = find_threshold_for_recall(y_test, y_proba_split)
print(f"Threshold terpilih (random split, target recall >= {TARGET_RECALL}): {best_threshold_split:.4f}")

idx = threshold_scan_split["threshold"].sub(best_threshold_split).abs().idxmin()
threshold_scan_split.iloc[max(0, idx - 3): idx + 4]

Threshold terpilih (random split, target recall >= 0.8): 0.4100


,threshold,recall,precision
37,0.38,0.826,0.328
38,0.39,0.826,0.339
39,0.40,0.826,0.358
40,0.41,0.826,0.373
41,0.42,0.739,0.347
42,0.43,0.739,0.354
43,0.44,0.739,0.354


### Cell 7 — Validasi Threshold pada Tiap Fold LOCO

**Tujuan**: Menguji ketahanan threshold terpilih pada skema LOCO, untuk memastikan model tetap memberikan recall yang wajar pada kota baru yang sama sekali tidak pernah diikutsertakan dalam pelatihan.

**Output yang diharapkan**:
- Performa model per kota menggunakan threshold hasil tuning serta nilai rata-rata Recall, Precision, dan F1.

In [7]:
def run_loco_with_threshold(df, feature_cols, threshold):
    cities = sorted(df[CITY_COL].unique())
    rows = []
    for city in cities:
        train_mask = df[CITY_COL] != city
        test_mask = df[CITY_COL] == city
        X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET_COL]
        X_test, y_test = df.loc[test_mask, feature_cols], df.loc[test_mask, TARGET_COL]

        model = make_rf().fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = (y_proba >= threshold).astype(int)

        rows.append({
            "held_out_city": city,
            "threshold": threshold,
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
        })
    return pd.DataFrame(rows)

loco_at_threshold = run_loco_with_threshold(df, feature_cols, best_threshold_split)
print(loco_at_threshold)
print("\nRata-rata pada threshold terpilih (skema LOCO):")
print(loco_at_threshold[["recall", "precision", "f1"]].mean().round(3))

  held_out_city  threshold    recall  precision        f1
0         ambon       0.41  0.800000   0.571429  0.666667
1           dki       0.41  0.935484   0.228346  0.367089
2       kebumen       0.41  0.000000   0.000000  0.000000
3     samarinda       0.41  0.857143   0.324324  0.470588

Rata-rata pada threshold terpilih (skema LOCO):
recall       0.648
precision    0.281
f1           0.376
dtype: float64


### Cell 8 — Model Final — Dilatih pada Seluruh Dataset

**Tujuan**: Melatih model Random Forest final menggunakan seluruh dataset (830 baris) agar model memiliki cakupan data spasial yang maksimal sebelum digunakan pada fase deployment.

In [8]:
rf_final = make_rf()
rf_final.fit(X, y)
print("Model final Random Forest dilatih pada seluruh dataset:", X.shape)

Model final Random Forest dilatih pada seluruh dataset: (830, 41)


### Cell 9 — SHAP (Global & Local Explanation)

**Tujuan**: Melakukan interpretabilitas model menggunakan SHAP TreeExplainer.

**Poin Penting**:
- Output SHAP dari Scikit-Learn `RandomForestClassifier` mengembalikan daftar (*list*) array untuk setiap kelas. Kita harus mengambil indeks `-1` (kelas 1 / slum) secara manual dengan pengecekan `isinstance(shap_values, list)`.
- Fungsi `explain_one_area` dibuat untuk mengurai faktor pendorong utama (key factors) lokal pada kelurahan/desa tertentu.

In [9]:
# Global SHAP
explainer = shap.TreeExplainer(rf_final)
shap_values = explainer.shap_values(X)

if isinstance(shap_values, list):
    shap_values = shap_values[-1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    if shap_values.shape[2] == 2:
        shap_values = shap_values[:, :, 1]
    elif shap_values.shape[0] == 2:
        shap_values = shap_values[1, :, :]

print("SHAP TreeExplainer berhasil dijalankan pada seluruh dataset.")

global_importance = pd.Series(
    np.abs(shap_values).mean(axis=0), index=feature_cols
).sort_values(ascending=False)

print("\nTop 10 Global Feature Importance (SHAP):")
print(global_importance.head(10))

SHAP TreeExplainer berhasil dijalankan pada seluruh dataset.

Top 10 Global Feature Importance (SHAP):
building_presence_mean              0.034712
b_coverage                          0.033215
building_presence_stdDev            0.032539
b_density_km2                       0.025141
ndbi_var_stdDev                     0.019426
vv_vh_stdDev                        0.017745
VV_stdDev                           0.016254
building_height_mean                0.015243
building_fractional_count_mean      0.014895
building_fractional_count_stdDev    0.014380
dtype: float64


In [10]:
# Fungsi Local SHAP per wilayah
def explain_one_area(idx: int) -> pd.DataFrame:
    row_meta = df.iloc[idx]
    row_features = X.iloc[idx]
    row_shap = shap_values[idx]
    y_proba_all = rf_final.predict_proba(X)[:, 1]
    proba = y_proba_all[idx]
    pred = int(proba >= best_threshold_split)

    print(f"Area: {row_meta['unit_name']} ({row_meta['city']}) | label asli: {'SLUM' if row_meta[TARGET_COL]==1 else 'NON-SLUM'} | "
          f"probabilitas model: {proba:.3f} | threshold: {best_threshold_split:.3f} | prediksi: {'SLUM' if pred==1 else 'NON-SLUM'}")

    expl_df = pd.DataFrame({
        "feature": feature_cols,
        "value": row_features.values,
        "shap_value": row_shap,
    })

    expl_df["arah"] = expl_df["shap_value"].apply(
        lambda s: "-> mendorong ke SLUM" if s > 0 else "-> menahan ke NON-SLUM"
    )
    return expl_df.reindex(expl_df["shap_value"].abs().sort_values(ascending=False).index).head(8)

print("\nContoh penjelasan local area (indeks 0):")
explain_one_area(0)


Contoh penjelasan local area (indeks 0):
Area: Kel. Nusaniwe (ambon) | label asli: NON-SLUM | probabilitas model: 0.765 | threshold: 0.410 | prediksi: SLUM


,feature,value,shap_value,arah
25,building_presence_mean,0.264990,0.036254,-> mendorong ke SLUM
32,b_coverage,0.316530,0.031445,-> mendorong ke SLUM
26,building_presence_stdDev,0.271335,0.031293,-> mendorong ke SLUM
33,b_density_km2,3537.732498,0.021285,-> mendorong ke SLUM
22,building_fractional_count_stdDev,0.001282,0.017849,-> mendorong ke SLUM
21,building_fractional_count_mean,0.001127,0.015623,-> mendorong ke SLUM
14,ndbi_var_stdDev,18.241799,0.014547,-> mendorong ke SLUM
23,building_height_mean,2.382818,0.014197,-> mendorong ke SLUM


### Cell 10 — Ekspor Artefak untuk Deployment

**Tujuan**: Menyimpan model Random Forest terlatih ke file `.joblib` dan metadatanya ke file `.json` agar siap dimuat oleh backend service.

**File Output**:
- `models/slum_rf_pipeline.joblib`
- `models/model_rf_metadata.json`

In [11]:
joblib.dump(rf_final, MODEL_PATH)
print(f"Model disimpan -> {MODEL_PATH}")

y_pred_tuned = (y_proba_split >= best_threshold_split).astype(int)
metadata = {
    "model_file": MODEL_PATH.name,
    "model_type": "RandomForestClassifier",
    "threshold": round(float(best_threshold_split), 4),
    "threshold_selection_method": f"max precision subject to recall >= {TARGET_RECALL} (random stratified split)",
    "feature_names": feature_cols,
    "n_features": len(feature_cols),
    "target_column": TARGET_COL,
    "cities": sorted(df[CITY_COL].unique().tolist()),
    "n_training_rows": int(len(df)),
    "class_balance": {
        "slum": int(y.sum()),
        "non_slum": int((y == 0).sum()),
        "pct_slum": round(float(y.mean() * 100), 2),
    },
    "validation_metrics": {
        "random_split": {
            "recall": round(float(recall_score(y_test, y_pred_tuned, zero_division=0)), 4),
            "precision": round(float(precision_score(y_test, y_pred_tuned, zero_division=0)), 4),
            "f1": round(float(f1_score(y_test, y_pred_tuned, zero_division=0)), 4),
            "roc_auc": round(float(roc_auc_score(y_test, y_proba_split)), 4),
        },
        "loco_at_threshold": {
            "recall_mean": round(float(loco_at_threshold["recall"].mean()), 4),
            "precision_mean": round(float(loco_at_threshold["precision"].mean()), 4),
            "f1_mean": round(float(loco_at_threshold["f1"].mean()), 4),
        },
        "loco_default_threshold_0_5": {
            "recall_mean": round(float(loco_summary.loc["mean", "recall"]), 4),
            "precision_mean": round(float(loco_summary.loc["mean", "precision"]), 4),
            "roc_auc_mean": round(float(loco_summary.loc["mean", "roc_auc"]), 4),
        },
    },
    "top_global_features": global_importance.head(10).round(4).to_dict(),
    "trained_on": str(date.today()),
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata disimpan -> {METADATA_PATH}")

Model disimpan -> models\slum_rf_pipeline.joblib
Metadata disimpan -> models\model_rf_metadata.json
